<a href="https://colab.research.google.com/github/J878-commits/Linkway-Learning-Python-Assignments/blob/main/Josy_Assignment_4___Pandas_Part_1_Fundamentals_(Practice).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📝 Assignment 4 — Pandas Part 1 (DataFrames, Filtering, Columns & Sorting)

Solve each question in the empty cell below it. Keep your code simple and clean.

* **Topics covered:** Loading CSVs, inspecting DataFrames, `mean` / `median` / `mode`, `.loc` & `.iloc`, boolean filtering (`&`, `|`, `isin`, `between`, `str.contains`), adding & modifying columns, and sorting (`sort_values`, `nlargest`).
* **Run the setup cell first** to generate the ShopSmart dataset (`sales.csv`).

In [1]:
import os

import numpy as np
import pandas as pd

OUT = "."
rng = np.random.default_rng(42)

products = pd.DataFrame(
    [
        ("P01", "Laptop", "Electronics", 55000, 46000),
        ("P02", "Smartphone", "Electronics", 22000, 17500),
        ("P03", "Headphones", "Electronics", 2500, 1400),
        ("P04", "Monitor", "Electronics", 12000, 9300),
        ("P05", "Keyboard", "Electronics", 1500, 800),
        ("P06", "Office Chair", "Furniture", 8500, 5600),
        ("P07", "Desk", "Furniture", 14000, 9800),
        ("P08", "Bookshelf", "Furniture", 6000, 3900),
        ("P09", "Filing Cabinet", "Furniture", 7500, 5200),
        ("P10", "Sofa", "Furniture", 32000, 24000),
        ("P11", "Notebook Pack", "Stationery", 250, 110),
        ("P12", "Pen Set", "Stationery", 180, 70),
        ("P13", "Printer Paper", "Stationery", 450, 300),
        ("P14", "Stapler", "Stationery", 300, 150),
        ("P15", "Whiteboard", "Stationery", 2200, 1300),
        ("P16", "T-Shirt", "Apparel", 600, 280),
        ("P17", "Jeans", "Apparel", 1800, 950),
        ("P18", "Jacket", "Apparel", 3500, 2100),
        ("P19", "Sneakers", "Apparel", 4200, 2600),
        ("P20", "Formal Shirt", "Apparel", 1400, 700),
    ],
    columns=["product_id", "product_name", "category", "unit_price", "unit_cost"],
)

cities = {
    "North": ["Delhi", "Chandigarh", "Jaipur", "Lucknow"],
    "South": ["Bengaluru", "Chennai", "Hyderabad", "Kochi"],
    "East": ["Kolkata", "Bhubaneswar", "Patna", "Guwahati"],
    "West": ["Mumbai", "Pune", "Ahmedabad", "Goa"],
}
first = ["Aarav", "Diya", "Rohan", "Isha", "Kabir", "Meera", "Arjun", "Sara", "Vikram", "Ananya",
         "Kunal", "Priya", "Nikhil", "Riya", "Aditya", "Neha", "Rahul", "Pooja", "Siddharth", "Tanvi"]
last = ["Sharma", "Iyer", "Das", "Patel", "Reddy", "Singh", "Mehta", "Nair", "Ghosh", "Kulkarni"]
n_cust = 150
regions = rng.choice(list(cities), size=n_cust, p=[0.3, 0.28, 0.17, 0.25])
customers = pd.DataFrame(
    {
        "customer_id": [f"C{i:03d}" for i in range(1, n_cust + 1)],
        "customer_name": [f"{rng.choice(first)} {rng.choice(last)}" for _ in range(n_cust)],
        "segment": rng.choice(["Consumer", "Corporate", "Small Business"], size=n_cust, p=[0.55, 0.25, 0.2]),
        "region": regions,
        "city": [rng.choice(cities[r]) for r in regions],
        "signup_date": pd.to_datetime("2023-01-01")
        + pd.to_timedelta(rng.integers(0, 700, size=n_cust), unit="D"),
    }
)

# orders: more orders in Q4 (festive season), growth in 2025
n_orders = 1200
days = pd.date_range("2024-01-01", "2025-12-31", freq="D")
weights = np.ones(len(days))
weights[days.month.isin([10, 11])] *= 1.8  # Diwali season
weights[days.month == 12] *= 1.4
weights[days.year == 2025] *= 1.25
weights /= weights.sum()
order_dates = np.sort(rng.choice(days, size=n_orders, p=weights))
cat_w = {"Electronics": 0.9, "Furniture": 0.6, "Stationery": 1.6, "Apparel": 1.3}
pw = products["category"].map(cat_w).to_numpy()
pw = pw / pw.sum()
prod_ids = rng.choice(products["product_id"], size=n_orders, p=pw)
cat_of = products.set_index("product_id")["category"]
qty = np.where(
    cat_of.loc[prod_ids].isin(["Stationery", "Apparel"]).to_numpy(),
    rng.integers(1, 11, size=n_orders),
    rng.integers(1, 4, size=n_orders),
)
orders = pd.DataFrame(
    {
        "order_id": [f"ORD{i:05d}" for i in range(10001, 10001 + n_orders)],
        "order_date": pd.to_datetime(order_dates),
        "customer_id": rng.choice(customers["customer_id"], size=n_orders),
        "product_id": prod_ids,
        "quantity": qty,
        "discount": rng.choice([0.0, 0.05, 0.10, 0.20], size=n_orders, p=[0.5, 0.25, 0.17, 0.08]),
        "payment_method": rng.choice(["UPI", "Credit Card", "Debit Card", "Cash on Delivery"],
                                     size=n_orders, p=[0.45, 0.25, 0.15, 0.15]),
        "status": rng.choice(["Delivered", "Returned", "Cancelled"], size=n_orders, p=[0.88, 0.07, 0.05]),
    }
)

sales = (
    orders.merge(customers[["customer_id", "segment", "region", "city"]], on="customer_id")
    .merge(products[["product_id", "product_name", "category", "unit_price", "unit_cost"]], on="product_id")
    .sort_values("order_id")
    .reset_index(drop=True)
)
sales = sales[["order_id", "order_date", "customer_id", "segment", "region", "city", "category",
               "product_name", "quantity", "unit_price", "unit_cost", "discount", "payment_method", "status"]]

# monthly targets per region
months = pd.period_range("2024-01", "2025-12", freq="M")
base = {"North": 210_000, "South": 175_000, "East": 95_000, "West": 150_000}
targets = pd.DataFrame(
    [(r, str(m), int(round(base[r] * (1.15 if m.year == 2025 else 1.0) * (1.5 if m.month in (10, 11) else 1.0), -4)))
     for r in base for m in months],
    columns=["region", "month", "target_revenue"],
)

# messy extract: 60 rows with typical real-world problems
messy = sales.sample(60, random_state=7).reset_index(drop=True)[
    ["order_id", "order_date", "region", "city", "category", "product_name", "quantity", "unit_price", "discount"]
].copy()
messy["order_date"] = messy["order_date"].dt.strftime("%Y-%m-%d").astype(object)
messy["unit_price"] = messy["unit_price"].astype(object)
messy["quantity"] = messy["quantity"].astype(object)
messy.loc[[2, 9, 17], "order_date"] = ["03/15/2024", "2024/07/04", "15-Aug-2025"]
messy.loc[[3, 11, 25, 40], "region"] = ["north", " South ", "WEST", "east "]
messy.loc[[5, 30], "city"] = ["bengaluru", "  Mumbai"]
messy.loc[[6, 21], "unit_price"] = ["₹1,500", "Rs. 600"]
messy.loc[[8, 33, 47], "quantity"] = [np.nan, "two", np.nan]
messy.loc[[12, 44], "discount"] = [np.nan, 5]  # 5 meaning 5% entered wrongly
messy.loc[[14], "category"] = [np.nan]
messy.loc[[19], "quantity"] = [-3]
messy.loc[[27], "unit_price"] = [9999999]
messy = pd.concat([messy, messy.iloc[[1, 10, 20]]], ignore_index=True)  # duplicates

products.to_csv(os.path.join(OUT, "products.csv"), index=False)
customers.to_csv(os.path.join(OUT, "customers.csv"), index=False)
orders.to_csv(os.path.join(OUT, "orders.csv"), index=False)
sales.to_csv(os.path.join(OUT, "sales.csv"), index=False)
targets.to_csv(os.path.join(OUT, "targets.csv"), index=False)
messy.to_csv(os.path.join(OUT, "messy_sales.csv"), index=False)
print("✅ ShopSmart dataset created:", ", ".join(f"{n}.csv" for n in ["sales","orders","customers","products","targets","messy_sales"]))

✅ ShopSmart dataset created: sales.csv, orders.csv, customers.csv, products.csv, targets.csv, messy_sales.csv


---
## Part A — Loading, Inspecting & Central Tendency (Q1–Q5)

**Q1.** Load `sales.csv` into a DataFrame `sales` (parsing `order_date` as dates using `parse_dates=["order_date"]`).
Print:
1. Shape of `sales` (`.shape`)
2. Column data types (`.dtypes`)
3. First 5 rows (`.head()`)

In [ ]:
# ✍️ Your answer
Shape of sales:
(1200, 14)

Data Types:
order_id                  object
order_date        datetime64[ns]
customer_id               object
segment                   object
region                    object
city                      object
category                  object
product_name              object
quantity                   int64
unit_price                 int64
unit_cost                  int64
discount                 float64
payment_method            object
status                    object
dtype: object

First 5 Rows:
   order_id order_date customer_id         segment region        city  \
0  ORD10001 2024-01-01        C103       Corporate  North      Jaipur
1  ORD10002 2024-01-02        C031       Corporate   East       Patna
2  ORD10003 2024-01-02        C011        Consumer  South       Kochi
3  ORD10004 2024-01-02        C121       Corporate   East     Kolkata
4  ORD10005 2024-01-03        C128  Small Business  North  Chandigarh

      category product_name  quantity  unit_price  unit_cost  discount  \
0   Stationery      Stapler         7         300        150       0.0
1   Stationery      Stapler         9         300        150       0.0
2  Electronics      Monitor         1       12000       9300       0.0
3   Stationery   Whiteboard         6        2200       1300       0.0
4   Stationery   Whiteboard         5        2200       1300       0.1

  payment_method     status
0    Credit Card  Delivered
1            UPI  Delivered
2    Credit Card  Delivered
3            UPI  Cancelled
4            UPI  Delivered


In [2]:
# ✍️ Your code here
import pandas as pd

# Load sales.csv and parse order_date as datetime
sales = pd.read_csv("sales.csv", parse_dates=["order_date"])

# 1. Shape of DataFrame
print("Shape of sales:")
print(sales.shape)

# 2. Column data types
print("\nData Types:")
print(sales.dtypes)

# 3. First 5 rows
print("\nFirst 5 Rows:")
print(sales.head())


Shape of sales:
(1200, 14)

Data Types:
order_id                  object
order_date        datetime64[ns]
customer_id               object
segment                   object
region                    object
city                      object
category                  object
product_name              object
quantity                   int64
unit_price                 int64
unit_cost                  int64
discount                 float64
payment_method            object
status                    object
dtype: object

First 5 Rows:
   order_id order_date customer_id         segment region        city  \
0  ORD10001 2024-01-01        C103       Corporate  North      Jaipur   
1  ORD10002 2024-01-02        C031       Corporate   East       Patna   
2  ORD10003 2024-01-02        C011        Consumer  South       Kochi   
3  ORD10004 2024-01-02        C121       Corporate   East     Kolkata   
4  ORD10005 2024-01-03        C128  Small Business  North  Chandigarh   

      category product_name  q

**Q2.** Check if any column in `sales` has missing values using `.isna().sum()`, and print the number of unique customers (`customer_id`), unique cities (`city`), and unique products (`product_name`) using `.nunique()`.

In [ ]:
# ✍️ Your answer
Missing Values:
order_id          0
order_date        0
customer_id       0
segment           0
region            0
city              0
category          0
product_name      0
quantity          0
unit_price        0
unit_cost         0
discount          0
payment_method    0
status            0
dtype: int64

Unique Customers: 150
Unique Cities: 16
Unique Products: 20

In [3]:
# ✍️ Your code here
# Check missing values in each column
print("Missing Values:")
print(sales.isna().sum())

# Count unique customers
print("\nUnique Customers:", sales["customer_id"].nunique())

# Count unique cities
print("Unique Cities:", sales["city"].nunique())

# Count unique products
print("Unique Products:", sales["product_name"].nunique())


Missing Values:
order_id          0
order_date        0
customer_id       0
segment           0
region            0
city              0
category          0
product_name      0
quantity          0
unit_price        0
unit_cost         0
discount          0
payment_method    0
status            0
dtype: int64

Unique Customers: 150
Unique Cities: 16
Unique Products: 20


**Q3.** For the `unit_price` and `quantity` columns in `sales`, calculate and print their **mean**, **median**, and **mode** (`.mode()[0]`).

In [ ]:
# ✍️ Your answer
Unit Price Statistics
Mean: 6847.4
Median: 1800.0
Mode: 450

Quantity Statistics
Mean: 4.12
Median: 3.0
Mode: 2

In [4]:
# ✍️ Your code here
# unit_price statistics
print("Unit Price Statistics")
print("Mean:", round(sales["unit_price"].mean(), 2))
print("Median:", sales["unit_price"].median())
print("Mode:", sales["unit_price"].mode()[0])

# quantity statistics
print("\nQuantity Statistics")
print("Mean:", round(sales["quantity"].mean(), 2))
print("Median:", sales["quantity"].median())
print("Mode:", sales["quantity"].mode()[0])


Unit Price Statistics
Mean: 6847.4
Median: 1800.0
Mode: 450

Quantity Statistics
Mean: 4.12
Median: 3.0
Mode: 2


**Q4.** Find the **mode** (most frequent value) of the categorical columns `region`, `category`, and `payment_method`, and print the full frequency table of `payment_method` (both raw counts and percentage share using `value_counts(normalize=True) * 100`).

In [ ]:
# ✍️ Your answer
Most Frequent Region: North
Most Frequent Category: Stationery
Most Frequent Payment Method: UPI

Payment Method Frequency (Counts):
payment_method
UPI                 543
Credit Card         303
Debit Card          184
Cash on Delivery    170
Name: count, dtype: int64

Payment Method Percentage Share (%):
payment_method
UPI                 45.25
Credit Card         25.25
Debit Card          15.33
Cash on Delivery    14.17
Name: proportion, dtype: float64

In [5]:
# ✍️ Your code here
# Mode of categorical columns
print("Most Frequent Region:", sales["region"].mode()[0])
print("Most Frequent Category:", sales["category"].mode()[0])
print("Most Frequent Payment Method:", sales["payment_method"].mode()[0])

# Payment method frequency (counts)
print("\nPayment Method Frequency (Counts):")
print(sales["payment_method"].value_counts())

# Payment method percentage share
print("\nPayment Method Percentage Share (%):")
print(round(sales["payment_method"].value_counts(normalize=True) * 100, 2))


Most Frequent Region: North
Most Frequent Category: Stationery
Most Frequent Payment Method: UPI

Payment Method Frequency (Counts):
payment_method
UPI                 543
Credit Card         303
Debit Card          184
Cash on Delivery    170
Name: count, dtype: int64

Payment Method Percentage Share (%):
payment_method
UPI                 45.25
Credit Card         25.25
Debit Card          15.33
Cash on Delivery    14.17
Name: proportion, dtype: float64


**Q5.** Check the order `status` breakdown (`Delivered`, `Returned`, `Cancelled`) using `.value_counts()` and print what percentage of total orders were `"Returned"` or `"Cancelled"`.

In [ ]:
# ✍️ Your answer
Order Status Breakdown:
status
Delivered    1033
Returned      103
Cancelled      64
Name: count, dtype: int64

Percentage of Returned or Cancelled Orders: 13.92 %

In [6]:
# ✍️ Your code here
# Status breakdown
status_counts = sales["status"].value_counts()

print("Order Status Breakdown:")
print(status_counts)

# Percentage of Returned or Cancelled orders
returned_cancelled_pct = (
    sales["status"].isin(["Returned", "Cancelled"]).mean() * 100
)

print(
    "\nPercentage of Returned or Cancelled Orders:",
    round(returned_cancelled_pct, 2),
    "%"
)


Order Status Breakdown:
status
Delivered    1033
Returned      103
Cancelled      64
Name: count, dtype: int64

Percentage of Returned or Cancelled Orders: 13.92 %


---
## Part B — Selecting (`.loc` / `.iloc`) & Filtering Rows (Q6–Q10)

**Q6.** Using `.iloc` and `.loc`:
1. Use `.iloc[:5, :4]` to display the first 5 rows and first 4 columns of `sales`.
2. Use `.loc[:4, ["order_id", "city", "product_name", "unit_price"]]` to display those 4 specific columns for the first 5 rows.

In [ ]:
# ✍️ Your answer
Using iloc:
   order_id order_date customer_id         segment
0  ORD10001 2024-01-01        C103       Corporate
1  ORD10002 2024-01-02        C031       Corporate
2  ORD10003 2024-01-02        C011        Consumer
3  ORD10004 2024-01-02        C121       Corporate
4  ORD10005 2024-01-03        C128  Small Business

Using loc:
   order_id        city product_name  unit_price
0  ORD10001      Jaipur      Stapler         300
1  ORD10002       Patna      Stapler         300
2  ORD10003       Kochi      Monitor       12000
3  ORD10004     Kolkata   Whiteboard        2200
4  ORD10005  Chandigarh   Whiteboard        2200

In [7]:
# ✍️ Your code here
# 1. First 5 rows and first 4 columns using iloc
print("Using iloc:")
print(sales.iloc[:5, :4])

# 2. Specific columns for first 5 rows using loc
print("\nUsing loc:")
print(
    sales.loc[:4, ["order_id", "city", "product_name", "unit_price"]]
)


Using iloc:
   order_id order_date customer_id         segment
0  ORD10001 2024-01-01        C103       Corporate
1  ORD10002 2024-01-02        C031       Corporate
2  ORD10003 2024-01-02        C011        Consumer
3  ORD10004 2024-01-02        C121       Corporate
4  ORD10005 2024-01-03        C128  Small Business

Using loc:
   order_id        city product_name  unit_price
0  ORD10001      Jaipur      Stapler         300
1  ORD10002       Patna      Stapler         300
2  ORD10003       Kochi      Monitor       12000
3  ORD10004     Kolkata   Whiteboard        2200
4  ORD10005  Chandigarh   Whiteboard        2200


**Q7.** Filter `sales` for orders where `region == "South"` **AND** `status == "Delivered"`.
Print how many orders match (`len(...)`) and the **mean** `unit_price` of those orders.

In [ ]:
# ✍️ Your answer
Number of Orders: 299
Mean Unit Price: 7340.13

In [8]:
# ✍️ Your code here
# Filter South region and Delivered orders
south_delivered = sales[
    (sales["region"] == "South") &
    (sales["status"] == "Delivered")
]

# Number of matching orders
print("Number of Orders:", len(south_delivered))

# Mean unit price
print(
    "Mean Unit Price:",
    round(south_delivered["unit_price"].mean(), 2)
)


Number of Orders: 299
Mean Unit Price: 7340.13


**Q8.** Filter `sales` for orders that meet **all** of these conditions:
* `category` is in `["Electronics", "Furniture"]` (use `.isin()`)
* `discount >= 0.10`
* `status == "Delivered"`

Print the number of matching orders and display the first 5 rows.

In [ ]:
# ✍️ Your answer
Number of Matching Orders: 69

First 5 Matching Rows:
     order_id order_date customer_id    segment region        city  \
13   ORD10014 2024-01-12        C029  Corporate  North  Chandigarh
35   ORD10036 2024-02-04        C011   Consumer  South       Kochi
40   ORD10041 2024-02-08        C071  Corporate  North  Chandigarh
99   ORD10100 2024-03-31        C121  Corporate   East     Kolkata
103  ORD10104 2024-04-02        C023   Consumer   West        Pune

        category product_name  quantity  unit_price  unit_cost  discount  \
13     Furniture         Desk         1       14000       9800       0.2
35   Electronics   Headphones         1        2500       1400       0.1
40   Electronics       Laptop         2       55000      46000       0.1
99   Electronics   Smartphone         3       22000      17500       0.1
103  Electronics      Monitor         1       12000       9300       0.1

       payment_method     status
13        Credit Card  Delivered
35   Cash on Delivery  Delivered
40        Credit Card  Delivered
99                UPI  Delivered
103        Debit Card  Delivered

In [9]:
# ✍️ Your code here
# Filter orders based on all three conditions
filtered_orders = sales[
    (sales["category"].isin(["Electronics", "Furniture"])) &
    (sales["discount"] >= 0.10) &
    (sales["status"] == "Delivered")
]

# Number of matching orders
print("Number of Matching Orders:", len(filtered_orders))

# First 5 rows
print("\nFirst 5 Matching Rows:")
print(filtered_orders.head())


Number of Matching Orders: 69

First 5 Matching Rows:
     order_id order_date customer_id    segment region        city  \
13   ORD10014 2024-01-12        C029  Corporate  North  Chandigarh   
35   ORD10036 2024-02-04        C011   Consumer  South       Kochi   
40   ORD10041 2024-02-08        C071  Corporate  North  Chandigarh   
99   ORD10100 2024-03-31        C121  Corporate   East     Kolkata   
103  ORD10104 2024-04-02        C023   Consumer   West        Pune   

        category product_name  quantity  unit_price  unit_cost  discount  \
13     Furniture         Desk         1       14000       9800       0.2   
35   Electronics   Headphones         1        2500       1400       0.1   
40   Electronics       Laptop         2       55000      46000       0.1   
99   Electronics   Smartphone         3       22000      17500       0.1   
103  Electronics      Monitor         1       12000       9300       0.1   

       payment_method     status  
13        Credit Card  Delivered 

**Q9.** Use `.between(5000, 25000)` on `unit_price` to filter for mid-priced products, and print the count of orders and the `.value_counts()` of `product_name` in that range.

In [ ]:
# ✍️ Your answer
Number of Orders: 231

Product Counts:
product_name
Smartphone        53
Monitor           52
Desk              39
Bookshelf         33
Office Chair      32
Filing Cabinet    22
Name: count, dtype: int64


In [11]:
# ✍️ Your code here
# Filter orders with unit_price between 5000 and 25000
mid_priced = sales[sales["unit_price"].between(5000, 25000)]

# Count of matching orders
print("Number of Orders:", len(mid_priced))

# Product frequency in this price range
print("\nProduct Counts:")
print(mid_priced["product_name"].value_counts())

Number of Orders: 231

Product Counts:
product_name
Smartphone        53
Monitor           52
Desk              39
Bookshelf         33
Office Chair      32
Filing Cabinet    22
Name: count, dtype: int64


**Q10.** Use `.str.contains("Shirt|Jacket", case=False)` on `product_name` to find all Shirt and Jacket orders. Print how many orders there are and the total `quantity` sold.

In [ ]:
# ✍️ Your answer
Number of Orders: 213
Total Quantity Sold: 1129

In [12]:
# ✍️ Your code here
# Filter orders containing "Shirt" or "Jacket" in product_name
shirt_jacket_orders = sales[
    sales["product_name"].str.contains("Shirt|Jacket", case=False)
]

# Number of orders
print("Number of Orders:", len(shirt_jacket_orders))

# Total quantity sold
print("Total Quantity Sold:", shirt_jacket_orders["quantity"].sum())


Number of Orders: 213
Total Quantity Sold: 1129


---
## Part C — Creating Columns & Sorting (Q11–Q15)

**Q11.** Add four new columns to `sales`:
1. `gross_revenue = sales["quantity"] * sales["unit_price"]`
2. `net_revenue = sales["gross_revenue"] * (1 - sales["discount"])`
3. `total_cost = sales["quantity"] * sales["unit_cost"]`
4. `profit = sales["net_revenue"] - sales["total_cost"]`

Print the **total**, **mean**, and **median** of `net_revenue` and `profit`.

In [ ]:
# ✍️ Your answer
Net Revenue Statistics
Total: 18931784.0
Mean: 15776.49
Median: 5820.0

Profit Statistics
Total: 4756264.0
Mean: 3963.55
Median: 2200.0

In [13]:
# ✍️ Your code here
# Create new columns
sales["gross_revenue"] = sales["quantity"] * sales["unit_price"]

sales["net_revenue"] = (
    sales["gross_revenue"] * (1 - sales["discount"])
)

sales["total_cost"] = sales["quantity"] * sales["unit_cost"]

sales["profit"] = (
    sales["net_revenue"] - sales["total_cost"]
)

# Net Revenue statistics
print("Net Revenue Statistics")
print("Total:", round(sales["net_revenue"].sum(), 2))
print("Mean:", round(sales["net_revenue"].mean(), 2))
print("Median:", round(sales["net_revenue"].median(), 2))

# Profit statistics
print("\nProfit Statistics")
print("Total:", round(sales["profit"].sum(), 2))
print("Mean:", round(sales["profit"].mean(), 2))
print("Median:", round(sales["profit"].median(), 2))

Net Revenue Statistics
Total: 18931784.0
Mean: 15776.49
Median: 5820.0

Profit Statistics
Total: 4756264.0
Mean: 3963.55
Median: 2200.0


**Q12.** Create a column `margin_pct = ((sales["profit"] / sales["net_revenue"]) * 100).round(2)` and a column `deal_size` using `np.where(sales["net_revenue"] >= 25000, "Large", "Regular")`.
Print `.value_counts()` of `deal_size`.

In [ ]:
# ✍️ Your answer
Deal Size Counts:
deal_size
Regular    986
Large      214
Name: count, dtype: int64

In [14]:
# ✍️ Your code here
# Create margin percentage column
sales["margin_pct"] = (
    (sales["profit"] / sales["net_revenue"]) * 100
).round(2)

# Create deal_size column
sales["deal_size"] = np.where(
    sales["net_revenue"] >= 25000,
    "Large",
    "Regular"
)

# Count of each deal size
print("Deal Size Counts:")
print(sales["deal_size"].value_counts())

Deal Size Counts:
deal_size
Regular    986
Large      214
Name: count, dtype: int64


**Q13.** Find the **Top 5 most profitable `"Delivered"` orders** using `.sort_values("profit", ascending=False)` (or `.nlargest(5, "profit")`), displaying only `["order_id", "order_date", "city", "product_name", "net_revenue", "profit"]`.

In [ ]:
# ✍️ Your answer
 order_id order_date        city product_name  net_revenue   profit
36   ORD10037 2024-02-04  Chandigarh       Laptop     165000.0  27000.0
110  ORD10111 2024-04-06       Patna       Laptop     165000.0  27000.0
230  ORD10231 2024-07-06   Hyderabad       Laptop     165000.0  27000.0
237  ORD10238 2024-07-10      Jaipur       Laptop     165000.0  27000.0
251  ORD10252 2024-07-18   Ahmedabad       Laptop     165000.0  27000.0

In [15]:
# ✍️ Your code here
# Top 5 most profitable delivered orders
top_5_profit = (
    sales[sales["status"] == "Delivered"]
    .nlargest(5, "profit")
)

print(
    top_5_profit[
        ["order_id", "order_date", "city",
         "product_name", "net_revenue", "profit"]
    ]
)


     order_id order_date        city product_name  net_revenue   profit
36   ORD10037 2024-02-04  Chandigarh       Laptop     165000.0  27000.0
110  ORD10111 2024-04-06       Patna       Laptop     165000.0  27000.0
230  ORD10231 2024-07-06   Hyderabad       Laptop     165000.0  27000.0
237  ORD10238 2024-07-10      Jaipur       Laptop     165000.0  27000.0
251  ORD10252 2024-07-18   Ahmedabad       Laptop     165000.0  27000.0


**Q14.** Sort `sales` by `region` (ascending) and `net_revenue` (descending) using `sales.sort_values(by=["region", "net_revenue"], ascending=[True, False])`, and display the top 5 rows with `["order_id", "region", "product_name", "net_revenue"]`.

In [ ]:
# ✍️ Your answer
order_id region product_name  net_revenue
110   ORD10111   East       Laptop     165000.0
1079  ORD11080   East       Laptop     165000.0
1013  ORD11014   East       Laptop     156750.0
279   ORD10280   East       Laptop     104500.0
77    ORD10078   East         Sofa      91200.0

In [16]:
# ✍️ Your code here
# Sort by region (ascending) and net_revenue (descending)
sorted_sales = sales.sort_values(
    by=["region", "net_revenue"],
    ascending=[True, False]
)

# Display top 5 rows
print(
    sorted_sales[
        ["order_id", "region", "product_name", "net_revenue"]
    ].head()
)


      order_id region product_name  net_revenue
110   ORD10111   East       Laptop     165000.0
1079  ORD11080   East       Laptop     165000.0
1013  ORD11014   East       Laptop     156750.0
279   ORD10280   East       Laptop     104500.0
77    ORD10078   East         Sofa      91200.0


**Q15.** Create a clean summary DataFrame `summary_df` by:
1. Dropping `"gross_revenue"` and `"total_cost"` using `.drop(columns=[...])`
2. Renaming `"net_revenue"` to `"revenue"` and `"payment_method"` to `"payment"` using `.rename(columns={...})`
Print `summary_df.columns.tolist()`.

In [ ]:
# ✍️ Your answer
['order_id', 'order_date', 'customer_id', 'segment', 'region', 'city', 'category', 'product_name', 'quantity', 'unit_price', 'unit_cost', 'discount', 'payment', 'status', 'revenue', 'profit', 'margin_pct', 'deal_size']


In [17]:
# ✍️ Your code here
# Create summary DataFrame
summary_df = (
    sales
    .drop(columns=["gross_revenue", "total_cost"])
    .rename(
        columns={
            "net_revenue": "revenue",
            "payment_method": "payment"
        }
    )
)

# Display column names
print(summary_df.columns.tolist())


['order_id', 'order_date', 'customer_id', 'segment', 'region', 'city', 'category', 'product_name', 'quantity', 'unit_price', 'unit_cost', 'discount', 'payment', 'status', 'revenue', 'profit', 'margin_pct', 'deal_size']
